# RB-LNL-Ti - Colab setup
Chạy notebook này một lần sau mỗi lần Colab tạo runtime mới. Nó cài dependency, mount Google Drive và kiểm tra môi trường; không chạy training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

candidates = [Path.cwd(), Path('/content/rb-lnl-ti'), Path('/content/drive/MyDrive/rb-lnl-ti')]
PROJECT_ROOT = next((p for p in candidates if (p / 'upgrade_models').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy project. Hãy clone/upload repo vào /content trước.')
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
!pip install -q -r upgrade_models/requirements.txt

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/rb-lnl-ti')
(DRIVE_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'submission' / 'results').mkdir(parents=True, exist_ok=True)
print('Persistent storage =', DRIVE_ROOT)

In [ ]:
import json
import subprocess
import sys
import torch
import torchvision
import timm
import yaml

assert torch.cuda.is_available(), 'Chưa thấy GPU. Hãy bật Runtime type = T4 GPU.'
subprocess.run([sys.executable, 'upgrade_models/train_colab.py', '--help'], check=True)
environment = {
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'timm': timm.__version__,
    'cuda': torch.version.cuda,
    'device': torch.cuda.get_device_name(0),
}
with (DRIVE_ROOT / 'environment.json').open('w', encoding='utf-8') as handle:
    json.dump(environment, handle, indent=2)
print(json.dumps(environment, indent=2))

## Chạy training
Sau notebook này, chạy lần lượt Stage 1 đến Stage 4. Các notebook stage đã dùng `config_colab.yaml`, nên checkpoint và kết quả được lưu trên Drive. Nếu runtime bị ngắt, chạy lại setup rồi dùng `--resume` ở stage đang dở.